# Notebook 10: A replicated aging signature (differential expression)

**Goal.** Which genes change with age, using the mouse as the unit of replication: sum counts per mouse, fit a model per gene, combine cohorts.

This replaces the descriptive fold-change ranking of the first guide. Cells from one mouse are added together first ('pseudobulk'), so the statistics see 19 mice, not 8,500 cells.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** **pseudobulk** sums the counts of all cells of one mouse. **age_de** converts to log2 counts-per-million and fits, for every gene, a straight line against log2(age); the sign and size of the slope become a z-score. It keeps genes with at least 1 CPM in half the mice. This is a light-weight alternative to DESeq2; for the final paper you can repeat it with pyDESeq2 (optional) and check the top genes agree.

*You should see:* Each cohort: 11 and 8 mice; all 1,500 genes tested; about 360 genes with p<0.05 (real data: 15,000+ genes tested).

In [ ]:
from mlc.dataset import Dataset
from mlc.analysis import pseudobulk, age_de, stouffer
from mlc import plots
sheet = pd.read_csv(SHEET).set_index("sample_id")
zs = {}
for c in cohorts()["mouse"]:
    ds = Dataset.load(PROC / f"{c}_scored")
    pb = pseudobulk(ds, MIN_CELLS)
    age = np.log2(sheet.loc[pb.columns, "age_months"].astype(float).values)
    de = age_de(pb, age); de.to_csv(RES / f"age_de_{c}.csv"); zs[c] = de["z"]
    print(c, "mice:", pb.shape[1], "| genes tested:", len(de), "| p<0.05:", int((de.p < 0.05).sum()))

**Cell 3.** **Stouffer's method** combines the cohorts' z-scores (sum divided by square root of the number of cohorts); **concordant** means every cohort moves in the same direction. Genes that are significant *and* concordant form the replicated aging signature.

*You should see:* DAM genes up with age (Itgax, Lpl, Cst7, Cd68, Trem2, Apoe; meta z about 6-7) and planted interface genes down (for example Bnip3).

In [ ]:
sig = stouffer(zs)
sig.to_csv(RES / "age_signature_meta.csv")
print(sig[sig.concordant].sort_values("p").head(10).round(4))

**Cell 4.** (a) The z-score of every gene in cohort 1 against cohort 2: genes along the diagonal replicate. Red = mitophagy and lysosome genes. (b) Volcano-style view of the combined signature.

In [ ]:
plots.fig_de(sig, G.MITOPHAGY + G.LYSOSOME); plt.show()